# Amazon Reviews MSGD Experiments (Plot-Only)

This notebook loads the latest saved batch results for a config and generates plots. It does not load datasets/embeddings or run simulations.

In [ ]:
%load_ext autoreload
%autoreload 2

from contextlib import contextmanager
from pathlib import Path
from types import SimpleNamespace
import os
import sys

import matplotlib.pyplot as plt

sys.path.append("./utils")

from experiment_io import load_experiment_config, load_latest_results


def _list_available_sweeps(results_payload):
    runs = results_payload.get("runs")
    if not runs:
        print("Payload has no sweep runs (legacy single-run format).")
        return
    print("Available sweeps:")
    for idx, run in enumerate(runs):
        print(f"  [{idx}] {run.get('sweep', {})}")


def _select_results_with_sweep(results_payload, sweep_filter=None):
    runs = results_payload.get("runs")
    if not runs:
        return results_payload, {}
    if sweep_filter is None:
        if len(runs) == 1:
            run = runs[0]
            return run["results"], dict(run.get("sweep") or {})
        _list_available_sweeps(results_payload)
        raise ValueError("Multiple sweep runs found. Set sweep_filter to one of the listed sweep dicts.")
    for run in runs:
        if run.get("sweep") == sweep_filter:
            return run["results"], dict(run.get("sweep") or {})
    _list_available_sweeps(results_payload)
    raise ValueError(f"No run found for sweep_filter={sweep_filter}")


def _build_plot_config(config_dict, selected_sweep=None):
    resolved = dict(config_dict)
    if selected_sweep:
        resolved.update(selected_sweep)
    resolved.setdefault("plot_seeds_separately", False)
    resolved.setdefault("plot_distance_separately", False)
    if "T" in resolved and "max_plot_iterations" not in resolved:
        resolved["max_plot_iterations"] = resolved["T"]
    if "fixed_probing_set" in resolved and "probing_set" not in resolved:
        resolved["probing_set"] = resolved["fixed_probing_set"]
    return SimpleNamespace(**resolved)


def load_plot_inputs(config_path, results_root=Path("results"), sweep_filter=None):
    dataset, config_dict, runner, raw_config = load_experiment_config(Path(config_path))
    results_payload, results_dir = load_latest_results(Path(config_path), results_root=Path(results_root))
    results_dict, selected_sweep = _select_results_with_sweep(results_payload, sweep_filter=sweep_filter)
    config = _build_plot_config(config_dict, selected_sweep)
    print(f"Dataset: {dataset}")
    print(f"Config: {config_path}")
    print(f"Results dir: {results_dir}")
    print(f"Selected sweep: {selected_sweep}")
    return dataset, config, runner, raw_config, results_payload, results_dir, results_dict, selected_sweep


@contextmanager
def plotting_io(save_plots=False, plot_output_dir=None):
    old_cwd = Path.cwd()
    orig_pyplot_savefig = plt.savefig
    fig_cls = type(plt.figure())
    plt.close('all')
    orig_fig_savefig = fig_cls.savefig

    def _noop_savefig(*args, **kwargs):
        return None

    try:
        if not save_plots:
            plt.savefig = _noop_savefig
            fig_cls.savefig = _noop_savefig
        elif plot_output_dir:
            target = Path(plot_output_dir)
            target.mkdir(parents=True, exist_ok=True)
            os.chdir(target)
        yield
    finally:
        plt.savefig = orig_pyplot_savefig
        fig_cls.savefig = orig_fig_savefig
        os.chdir(old_cwd)

from utils_plotting_amazon import (
    plot_individual_model_accuracies,
    plot_individual_model_accuracies_with_kappa,
    plot_assignment_fraction,
    plot_final_accuracy_vs_p,
)


## Configuration

In [ ]:
config_path = Path("configs/amazon_bad.yaml")
results_root = Path("results")
sweep_filter = None  # Example: {"eta": 0.05}
SAVE_PLOTS = False
PLOT_OUTPUT_DIR = None  # Example: Path("submission_figures/amazon")


## Load Latest Results

In [ ]:
dataset, config, runner, raw_config, results_payload, results_dir, results_dict, selected_sweep = load_plot_inputs(
    config_path=config_path,
    results_root=results_root,
    sweep_filter=sweep_filter,
)
if dataset != "amazon":
    raise ValueError(f"Expected amazon config, got {dataset}")


## Print Final Results Summary

In [ ]:
def summarize_classification_results(results_dict):
    print("\n" + "=" * 80)
    print("FINAL RESULTS SUMMARY (results-only)")
    print("=" * 80)
    for key in sorted(results_dict.keys()):
        res = results_dict[key]
        final_msgd = res["model_acc"][-1]
        final_update_all = res.get("model_acc_full", [None])[-1]
        final_ensemble = res.get("ensemble_acc", [None])[-1]
        print(f"\nRun {key}:")
        print(f"  MSGD:       avg={final_msgd.mean():.4f}, min={final_msgd.min():.4f}, max={final_msgd.max():.4f}")
        if final_update_all is not None:
            print(f"  Update_all: avg={final_update_all.mean():.4f}, min={final_update_all.min():.4f}, max={final_update_all.max():.4f}")
        if final_ensemble is not None:
            try:
                print(f"  Ensemble:   {float(final_ensemble):.4f}")
            except Exception:
                print(f"  Ensemble:   {final_ensemble}")
    print("\n" + "=" * 80)

summarize_classification_results(results_dict)


## Generate Plots

In [ ]:
with plotting_io(save_plots=SAVE_PLOTS, plot_output_dir=PLOT_OUTPUT_DIR):
    plot_individual_model_accuracies(
        results_dict,
        config,
        only_p0=False,
        show_title=False,
        show_legend=True,
        save_file_name=("amazon_msgd_results" if SAVE_PLOTS else None),
        baseline_acc=None,
    )


In [ ]:
with plotting_io(save_plots=SAVE_PLOTS, plot_output_dir=PLOT_OUTPUT_DIR):
    plot_final_accuracy_vs_p(
        results_dict,
        config,
        save_file_name=("amazon_final_accuracy_vs_p" if SAVE_PLOTS else None),
        baseline_acc=None,
        flag_single_row=True,
        flag_kappa=True,
    )


In [ ]:
kappas = sorted({k for (_, _, k) in results_dict.keys()})
if len(kappas) > 1:
    with plotting_io(save_plots=SAVE_PLOTS, plot_output_dir=PLOT_OUTPUT_DIR):
        plot_individual_model_accuracies_with_kappa(results_dict, title=None)
else:
    print("Skipping kappa plot: only one kappa value present.")


In [ ]:
with plotting_io(save_plots=SAVE_PLOTS, plot_output_dir=PLOT_OUTPUT_DIR):
    plot_assignment_fraction(
        results_dict,
        config,
        title=None,
        max_iter=getattr(config, "max_plot_iterations", None),
    )


## Notes

- This notebook intentionally omits embedding generation, data loading, clustering, baselines, and user-assignment-map plots.
- If your results payload contains multiple sweep runs, set `sweep_filter` to the exact sweep dict shown in the load step.